# LLM Systems Lab: PyTorch Profiler Trace Export
This notebook runs the PyTorch Profiler natively and exports traces for TensorBoard.

In [ ]:
import torch
from torch.profiler import profile, record_function, ProfilerActivity
from gpu_systems.model_configs import load_all_configs
from gpu_systems.benchmark import benchmark_model

configs = load_all_configs()
config = configs.get("medium", list(configs.values())[0])
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

print(f"Using device: {device}")
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA] if device == "cuda" else [ProfilerActivity.CPU],
    schedule=torch.profiler.schedule(wait=0, warmup=1, active=3, repeat=1),
    on_trace_ready=torch.profiler.tensorboard_trace_handler('./log_dir_forward'),
    record_shapes=True,
    profile_memory=True,
    with_stack=True
) as prof:
    with record_function("forward_pass_benchmark"):
        benchmark_model(configs_dict=config, w=1, n=3, mode="forward", device=device)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))